# Build Lineage Views
Create SQL views/tables that connect **Semantic Models** with **Report Elements** for comprehensive lineage analysis.

## Purpose
This notebook addresses the challenges of linking reports to semantic model objects:
1. **Report visuals** use `Page Name` + `Visual Name` as composite identifiers (no single GUID)
2. **list_semantic_model_objects** returns **report-level** usage (not visual-level granularity)
3. Returns column/measure **names** (not lineage tags), so we join by name and enrich with lineage tags

## Data Sources
- **Primary**: `lineage_report_visuals` (always required for visual metadata)
- **Optional**: `lineage_report_pages` (often empty - pages derived from visuals if missing)
- **Semantic Models**: `lineage_semantic_models`, `lineage_semantic_model_columns`, `lineage_semantic_model_measures`
- **Report Usage**: `lineage_report_semantic_model_objects` (report-level object usage)

## Output Views

### Simple Enrichment Views (for frontend performance)
- `vw_report_visuals_enriched` - Visuals with **dataset_id** pre-joined from parent report
- `vw_report_pages_enriched` - Pages **derived from visuals** with **dataset_id** pre-joined
- ✅ **Fast queries**: Pre-computed joins eliminate frontend lookups
- ✅ **Resilient**: Pages derived from visuals when `lineage_report_pages` table doesn't exist
- ⚠️ **Limited fields**: Only includes dataset_id, not dataset_name (for names, use complex views)

### Advanced Analysis Views (for complex queries)
- `v_report_to_dataset_summary` - Report-to-dataset relationship summary with counts and **dataset_name**
- `v_report_to_semantic_model` - Maps **reports** to semantic model columns/measures with lineage tags and **dataset_name**
- ✅ **Rich metadata**: Includes dataset names via join to lineage_semantic_models
- ✅ **Lineage tags**: Enriched with column/measure lineage tags for dependency tracking
- ⚠️ **Report-level only**: Shows which objects are used in reports, not which specific visuals use them

**Note**: Semantic model lineage and complete field-level lineage calculations are performed in downstream processing.

## 1. Setup and Imports

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = globals().get("spark")
if spark is None:
    spark = SparkSession.builder.getOrCreate()

print("✅ Imports successful")

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 9, Finished, Available, Finished, False)

✅ Imports successful


## 2. Configuration

In [ ]:
# Configuration for view creation
VIEW_MODE = "create_or_replace"  # "create_or_replace" or "drop_and_create"

# Source tables (created by extraction notebooks)
SOURCE_TABLES = [
    "lineage_semantic_models",
    "lineage_semantic_model_columns", 
    "lineage_semantic_model_measures",
    "lineage_semantic_model_dependencies",
    "lineage_reports",
    "lineage_report_visual_objects",
    "lineage_report_semantic_model_objects",
    "lineage_report_visuals"
    # NOTE: lineage_report_pages excluded - often empty, pages derived from visuals instead
]

# Verify all source tables exist
print("Checking source tables...")
missing_tables = []
for table_name in SOURCE_TABLES:
    if spark.catalog.tableExists(table_name):
        count = spark.table(table_name).count()
        print(f"  ✅ {table_name}: {count} rows")
    else:
        print(f"  ❌ {table_name}: NOT FOUND")
        missing_tables.append(table_name)

if missing_tables:
    print(f"\n⚠️  WARNING: {len(missing_tables)} tables missing. Run extraction notebooks first:")
    for table in missing_tables:
        if "semantic_model" in table:
            print("     - 01_extract_semantic_models.ipynb")
        elif "report" in table:
            print("     - 02_extract_reports.ipynb")
else:
    print(f"\n✅ All {len(SOURCE_TABLES)} source tables found")
    
# Check if lineage_report_pages exists (optional)
if spark.catalog.tableExists("lineage_report_pages"):
    page_count = spark.table("lineage_report_pages").count()
    print(f"\n💡 Optional table found: lineage_report_pages ({page_count} rows)")
else:
    print(f"\n💡 Optional table lineage_report_pages not found - pages will be derived from visuals")

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 10, Finished, Available, Finished, False)

Checking source tables...
  ✅ lineage_semantic_models: 6 rows
  ✅ lineage_semantic_model_columns: 337 rows
  ✅ lineage_semantic_model_measures: 46 rows
  ✅ lineage_semantic_model_dependencies: 950 rows
  ✅ lineage_reports: 1 rows
  ✅ lineage_report_visual_objects: 67 rows
  ✅ lineage_report_semantic_model_objects: 96 rows
  ✅ lineage_report_visuals: 36 rows

✅ All 8 source tables found


## 2a. Inspect Table Schemas
Check actual columns in source tables to ensure views match the schema.

**Important**: Run this cell first to verify table structures before creating views.

In [ ]:
# Inspect schemas of key tables for view creation
print("📊 Table Schemas:\n")

for table in ["lineage_report_visuals", "lineage_reports", "lineage_report_semantic_model_objects"]:
    if spark.catalog.tableExists(table):
        df = spark.table(table)
        print(f"\n{table}:")
        print(f"  Columns: {', '.join(df.columns)}")
        print(f"  Rows: {df.count()}")
        print(f"  Sample row:")
        display(df.limit(1))
    else:
        print(f"\n{table}: NOT FOUND")

# Deep dive into semantic model objects to understand report_source fields
if spark.catalog.tableExists("lineage_report_semantic_model_objects"):
    print("\n🔍 DEEP DIVE: lineage_report_semantic_model_objects")
    df = spark.table("lineage_report_semantic_model_objects")
    print("\nSample rows with report_source and report_source_object:")
    display(df.select("object_name", "object_type", "table_name", "report_source", "report_source_object").limit(5))
    
    print("\nDistinct report_source values:")
    display(df.select("report_source").distinct())
        
# Check optional tables
if spark.catalog.tableExists("lineage_report_pages"):
    df = spark.table("lineage_report_pages")
    print(f"\n[OPTIONAL] lineage_report_pages:")
    print(f"  Columns: {', '.join(df.columns)}")
    print(f"  Rows: {df.count()}")
else:
    print(f"\n[OPTIONAL] lineage_report_pages: NOT FOUND (pages will be derived from visuals)")

## 2b. Quick Diagnostic - Check What Exists
Run this cell to see what tables and views currently exist in your lakehouse.

In [ ]:
print("🔍 DIAGNOSTIC: Checking what exists in this lakehouse...\n")

# Check for base tables
base_tables = [
    "lineage_reports",
    "lineage_report_pages", 
    "lineage_report_visuals",
    "lineage_report_semantic_model_objects",
    "lineage_semantic_models",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_measures"
]

print("📦 Base Tables:")
for table in base_tables:
    if spark.catalog.tableExists(table):
        count = spark.table(table).count()
        print(f"  ✅ {table}: {count} rows")
    else:
        print(f"  ❌ {table}: NOT FOUND")

# Check for views
views = [
    "vw_report_visuals_enriched",
    "vw_report_pages_enriched", 
    "v_report_to_dataset_summary",
    "v_report_to_semantic_model"
]

print("\n📊 Views (should be created by this notebook):")
for view in views:
    if spark.catalog.tableExists(view):
        count = spark.table(view).count()
        print(f"  ✅ {view}: {count} rows")
    else:
        print(f"  ❌ {view}: NOT CREATED YET")

# List all tables/views in the current database
print("\n📋 All tables/views in current database:")
all_tables = spark.sql("SHOW TABLES").collect()
for row in all_tables:
    table_name = row.tableName
    is_temp = row.isTemporary
    print(f"  - {table_name} {'(temp)' if is_temp else ''}")

print("\n💡 If base tables show ❌, run extraction notebooks first:")
print("   - 01_extract_semantic_models.ipynb")
print("   - 02_extract_reports.ipynb")
print("\n💡 If views show ❌, continue running cells below to create them")

## 3. Create View: Report to Dataset Mapping
Maps reports to their semantic models (datasets).

In [9]:
%%sql
CREATE OR REPLACE VIEW v_report_to_dataset_summary AS
SELECT 
    r.workspace_id,
    r.report_id,
    r.report_name,
    r.dataset_id,
    r.extraction_timestamp AS report_extraction_timestamp,
    r.count_pages,
    r.count_visuals,
    r.count_visualobjects AS count_visual_objects,
    r.count_semanticmodelobjects AS count_semantic_model_objects,
    
    -- Semantic model details
    sm.model_name AS dataset_name,
    sm.extraction_timestamp AS dataset_extraction_timestamp,
    sm.count_tables,
    sm.count_measures,
    sm.count_columns,
    sm.count_relationships,
    sm.count_dependencies

FROM lineage_reports r

LEFT JOIN lineage_semantic_models sm
    ON r.dataset_id = sm.model_id
    AND r.workspace_id = sm.workspace_id

WHERE r.dataset_id IS NOT NULL

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## 3a. Create Simple Enrichment Views (Basic Joins)
These views add dataset_id to visuals and pages for simple lookups. Used by frontend code.

**Schema Note**: 
- Views use actual columns from extraction tables (run cell 2a to inspect)
- Column names match the output from `sempy.report.list_visuals()` and `list_pages()`
- Only `dataset_id` is included (not dataset_name) - for dataset names, use the complex views or join to `lineage_semantic_models`

**Pages Derivation**:
- `vw_report_pages_enriched` derives pages from `lineage_report_visuals` using `page_name`
- This handles cases where `lineage_report_pages` table doesn't exist or is empty
- Matches the frontend approach: extract unique page names from visuals

In [ ]:
%%sql
-- Simple view: Visuals enriched with dataset_id from parent report
-- Used by frontend to avoid manual joins in JavaScript
CREATE OR REPLACE VIEW vw_report_visuals_enriched AS
SELECT 
    v.visual_name,
    v.title,
    v.type,
    v.url,
    v.report_id,
    v.report_name,
    v.workspace_id,
    v.extraction_timestamp,
    -- Enriched fields from parent report
    r.dataset_id,
    r.report_name AS parent_report_name
FROM lineage_report_visuals v
LEFT JOIN lineage_reports r 
    ON v.report_id = r.report_id
    AND v.workspace_id = r.workspace_id

### Alternative: Python API (use if %%sql magic fails)

In [ ]:
# Alternative approach if %%sql magic doesn't work
# Run this cell instead of the %%sql cells above

print("Creating all views using spark.sql() API...\n")

# View 1: vw_report_visuals_enriched
spark.sql("""
CREATE OR REPLACE VIEW vw_report_visuals_enriched AS
SELECT 
    v.visual_name,
    v.title,
    v.type,
    v.url,
    v.report_id,
    v.report_name,
    v.workspace_id,
    v.extraction_timestamp,
    r.dataset_id,
    r.report_name AS parent_report_name
FROM lineage_report_visuals v
LEFT JOIN lineage_reports r 
    ON v.report_id = r.report_id
    AND v.workspace_id = r.workspace_id
""")
print("✅ vw_report_visuals_enriched created")

# View 2: vw_report_pages_enriched
spark.sql("""
CREATE OR REPLACE VIEW vw_report_pages_enriched AS
SELECT DISTINCT
    v.page_name,
    v.page_name AS display_name,
    NULL AS page_number,
    v.report_id,
    v.report_name,
    v.workspace_id,
    MIN(v.extraction_timestamp) AS extraction_timestamp,
    r.dataset_id,
    r.report_name AS parent_report_name
FROM lineage_report_visuals v
LEFT JOIN lineage_reports r 
    ON v.report_id = r.report_id
    AND v.workspace_id = r.workspace_id
WHERE v.page_name IS NOT NULL
GROUP BY v.page_name, v.report_id, v.report_name, v.workspace_id, r.dataset_id, r.report_name
""")
print("✅ vw_report_pages_enriched created")

# View 3: v_report_to_dataset_summary
spark.sql("""
CREATE OR REPLACE VIEW v_report_to_dataset_summary AS
SELECT 
    r.workspace_id,
    r.report_id,
    r.report_name,
    r.dataset_id,
    r.extraction_timestamp AS report_extraction_timestamp,
    r.count_pages,
    r.count_visuals,
    r.count_visualobjects AS count_visual_objects,
    r.count_semanticmodelobjects AS count_semantic_model_objects,
    sm.model_name AS dataset_name,
    sm.extraction_timestamp AS dataset_extraction_timestamp,
    sm.count_tables,
    sm.count_measures,
    sm.count_columns,
    sm.count_relationships,
    sm.count_dependencies
FROM lineage_reports r
LEFT JOIN lineage_semantic_models sm
    ON r.dataset_id = sm.model_id
    AND r.workspace_id = sm.workspace_id
WHERE r.dataset_id IS NOT NULL
""")
print("✅ v_report_to_dataset_summary created")

# View 4: v_report_to_semantic_model
spark.sql("""
CREATE OR REPLACE VIEW v_report_to_semantic_model AS
SELECT
    rsmo.workspace_id,
    rsmo.report_id,
    rsmo.report_name,
    rsmo.extraction_timestamp,
    rsmo.object_name,
    rsmo.object_type,
    rsmo.table_name,
    rsmo.report_source,
    rsmo.report_source_object,
    r.dataset_id AS model_id,
    r.dataset_id,
    sm.model_name AS dataset_name,
    col.lineagetag AS column_lineage_tag,
    col.datatype AS column_datatype,
    col.description AS column_description,
    col.ishidden AS column_is_hidden,
    col.sourcecolumn AS column_source,
    col.sourcelineagetag AS column_source_lineage_tag,
    msr.lineagetag AS measure_lineage_tag,
    msr.expression AS measure_expression,
    msr.formatstring AS measure_format_string,
    msr.description AS measure_description,
    COALESCE(col.lineagetag, msr.lineagetag) AS lineage_tag,
    CASE 
        WHEN col.lineagetag IS NOT NULL THEN 'Column'
        WHEN msr.lineagetag IS NOT NULL THEN 'Measure'
        ELSE rsmo.object_type
    END AS enriched_object_type
FROM lineage_report_semantic_model_objects rsmo
INNER JOIN lineage_reports r
    ON rsmo.report_id = r.report_id
    AND rsmo.workspace_id = r.workspace_id
LEFT JOIN lineage_semantic_models sm
    ON r.dataset_id = sm.model_id
    AND r.workspace_id = sm.workspace_id
LEFT JOIN lineage_semantic_model_columns col
    ON r.dataset_id = col.model_id
    AND rsmo.workspace_id = col.workspace_id
    AND rsmo.object_name = col.name
    AND rsmo.table_name = col.table
LEFT JOIN lineage_semantic_model_measures msr
    ON r.dataset_id = msr.model_id
    AND rsmo.workspace_id = msr.workspace_id
    AND rsmo.object_name = msr.name
    AND rsmo.table_name = msr.table
WHERE r.dataset_id IS NOT NULL
""")
print("✅ v_report_to_semantic_model created")

print("\n🎉 All 4 views created successfully!")
print("\n📊 View row counts:")
for view_name in ["vw_report_visuals_enriched", "vw_report_pages_enriched", 
                   "v_report_to_dataset_summary", "v_report_to_semantic_model"]:
    count = spark.table(view_name).count()
    print(f"  {view_name}: {count:,} rows")

In [ ]:
%%sql
-- Simple view: Pages enriched with dataset_id from parent report
-- NOTE: Derives pages from visuals since lineage_report_pages may not exist
CREATE OR REPLACE VIEW vw_report_pages_enriched AS
SELECT DISTINCT
    v.page_name,
    v.page_name AS display_name,  -- Use page_name as display_name
    NULL AS page_number,  -- Page number not available when deriving from visuals
    v.report_id,
    v.report_name,
    v.workspace_id,
    MIN(v.extraction_timestamp) AS extraction_timestamp,
    -- Enriched fields from parent report
    r.dataset_id,
    r.report_name AS parent_report_name
FROM lineage_report_visuals v
LEFT JOIN lineage_reports r 
    ON v.report_id = r.report_id
    AND v.workspace_id = r.workspace_id
WHERE v.page_name IS NOT NULL
GROUP BY v.page_name, v.report_id, v.report_name, v.workspace_id, r.dataset_id, r.report_name

In [ ]:
print("✅ Simple enrichment views created:")
print(f"   vw_report_visuals_enriched: {spark.table('vw_report_visuals_enriched').count()} rows")
print(f"   vw_report_pages_enriched: {spark.table('vw_report_pages_enriched').count()} rows")
print("\n💡 These views simplify frontend queries by pre-joining dataset_id")
display(spark.table('vw_report_visuals_enriched').limit(3))

In [10]:
print("✅ View created: v_report_to_dataset_summary")
print(f"   Rows: {spark.table('v_report_to_dataset_summary').count()}")
display(spark.table('v_report_to_dataset_summary').limit(5))

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 12, Finished, Available, Finished, False)

✅ View created: v_report_to_dataset_summary
   Rows: 1


SynapseWidget(Synapse.DataFrame, 57be15f1-0204-41a7-91bd-15dd73092aa7)

## 4. Create View: Report to Semantic Model Objects
**CRITICAL MAPPING**: Maps semantic model objects (columns/measures) used in reports.

**Data Source**: The `lineage_report_semantic_model_objects` table is created by `list_semantic_model_objects()` which extracts:
- Which columns and measures are referenced in the report
- The semantic model (dataset_id via report join)
- Report-level usage tracking

**Schema Note**: 
- This table contains **report-level** object usage, not visual-level granularity
- Fields: `object_name`, `object_type`, `table_name`, `report_source`, `report_source_object`
- NO `page_name` or `visual_name` columns - those are in `lineage_report_visuals` instead
- `report_source` and `report_source_object` may contain additional context about usage location

**Note**: `list_semantic_model_objects` returns column/measure **names** (not lineage tags), so we join by name and then enrich with lineage tags from the semantic model metadata tables.

In [ ]:
%%sql
CREATE OR REPLACE VIEW v_report_to_semantic_model AS
-- Note: This view maps semantic model objects to reports (report-level, not visual-level)
-- The lineage_report_semantic_model_objects table contains report-level usage
-- We enrich it with metadata from semantic model tables and add lineage tags

SELECT
    -- Report and object identifiers
    rsmo.workspace_id,
    rsmo.report_id,
    rsmo.report_name,
    rsmo.extraction_timestamp,
    rsmo.object_name,
    rsmo.object_type,
    rsmo.table_name,
    
    -- Usage context (may contain page/visual info in structured format)
    rsmo.report_source,
    rsmo.report_source_object,
    
    -- Get model_id and dataset_id from reports table
    r.dataset_id AS model_id,
    r.dataset_id,
    sm.model_name AS dataset_name,
    
    -- Enrich with semantic model column metadata (if it's a column)
    col.lineagetag AS column_lineage_tag,
    col.datatype AS column_datatype,
    col.description AS column_description,
    col.ishidden AS column_is_hidden,
    col.sourcecolumn AS column_source,
    col.sourcelineagetag AS column_source_lineage_tag,
    
    -- Enrich with semantic model measure metadata (if it's a measure)
    msr.lineagetag AS measure_lineage_tag,
    msr.expression AS measure_expression,
    msr.formatstring AS measure_format_string,
    msr.description AS measure_description,
    
    -- Computed: final lineage tag (column or measure)
    COALESCE(col.lineagetag, msr.lineagetag) AS lineage_tag,
    
    -- Computed: object category
    CASE 
        WHEN col.lineagetag IS NOT NULL THEN 'Column'
        WHEN msr.lineagetag IS NOT NULL THEN 'Measure'
        ELSE rsmo.object_type
    END AS enriched_object_type

FROM lineage_report_semantic_model_objects rsmo

INNER JOIN lineage_reports r
    ON rsmo.report_id = r.report_id
    AND rsmo.workspace_id = r.workspace_id

-- Join to semantic models to get dataset name
LEFT JOIN lineage_semantic_models sm
    ON r.dataset_id = sm.model_id
    AND r.workspace_id = sm.workspace_id

-- Join to semantic model columns by NAME (list_semantic_model_objects uses names, not lineage tags)
-- Use dataset_id from reports table as model_id
LEFT JOIN lineage_semantic_model_columns col
    ON r.dataset_id = col.model_id
    AND rsmo.workspace_id = col.workspace_id
    AND rsmo.object_name = col.name
    AND rsmo.table_name = col.table

-- Join to semantic model measures by NAME
LEFT JOIN lineage_semantic_model_measures msr
    ON r.dataset_id = msr.model_id
    AND rsmo.workspace_id = msr.workspace_id
    AND rsmo.object_name = msr.name
    AND rsmo.table_name = msr.table

WHERE r.dataset_id IS NOT NULL

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
print("✅ View created: v_report_to_semantic_model")
print(f"   Rows: {spark.table('v_report_to_semantic_model').count()}")
print("\n📋 Sample data:")
display(spark.table('v_report_to_semantic_model').limit(10))

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 14, Finished, Available, Finished, False)

✅ View created: v_report_visual_to_semantic_model
   Rows: 3456


SynapseWidget(Synapse.DataFrame, db95f253-c3fb-425d-a57b-dd312806d6f9)

## 5. Summary
All lineage views created successfully!

**Note**: `vw_report_pages_enriched` derives pages from visuals using `GROUP BY page_name`. This approach:
- ✅ Works when `lineage_report_pages` table doesn't exist
- ✅ Matches the frontend implementation
- ✅ Ensures page nodes appear even when extraction didn't capture pages directly

In [ ]:
print("\n🎉 All lineage views created successfully!\n")
print("=" * 60)

views = [
    "vw_report_visuals_enriched",
    "vw_report_pages_enriched",
    "v_report_to_dataset_summary",
    "v_report_to_semantic_model"
]

for view_name in views:
    count = spark.table(view_name).count()
    print(f"✅ {view_name}: {count:,} rows")

print("=" * 60)
print("\n💡 Simple views (vw_*): Used by frontend for fast lookups")
print("💡 Complex views (v_*): Used for advanced lineage analysis")
print("\n⚠️  Note: v_report_to_semantic_model contains REPORT-LEVEL usage")
print("   Visual-level lineage requires parsing report_source fields")

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 15, Finished, Available, Finished, False)


🎉 All lineage views created successfully!

✅ v_report_to_dataset_summary: 1 rows
✅ v_report_visual_to_semantic_model: 3,456 rows

💡 Use these views for lineage analysis and reporting


## 6. Example Query: Find reports using a semantic model
Shows all reports connected to a specific semantic model

In [14]:
%%sql
-- Example: Find all reports using a specific semantic model
SELECT 
    r.report_name,
    r.dataset_name,
    r.count_pages,
    r.count_visuals,
    r.count_visual_objects,
    r.count_semantic_model_objects
FROM v_report_to_dataset_summary r
WHERE r.dataset_name = '<your-dataset-name>'
ORDER BY r.report_name
LIMIT 100

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 6 fields>

## 7. Example Query: Find all reports using a specific semantic model column
Shows which reports consume a particular semantic model field (report-level, not visual-level)

In [ ]:
%%sql
-- Example: Find all reports using a specific column from a semantic model
SELECT DISTINCT
    report_name,
    object_name,
    object_type,
    table_name,
    lineage_tag,
    report_source,
    report_source_object
FROM v_report_to_semantic_model
WHERE object_name = '<your-column-name>'
  AND table_name = '<your-table-name>'
ORDER BY report_name
LIMIT 100

StatementMeta(, cf3c2a14-2238-41f1-855d-363c8567d41c, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 5 fields>

## Key Notes

### Report-Level vs Visual-Level Granularity
- **`lineage_report_semantic_model_objects`** contains **REPORT-LEVEL** usage data
- It shows which columns/measures are referenced in a report, but NOT which specific visual uses them
- Visual-level granularity would require parsing `report_source` and `report_source_object` fields
- For visual-level lineage, use `lineage_report_visuals` joined with visual-specific metadata

### Report Visual Identifiers
- Report visuals do NOT have a single GUID/ID column
- They are identified by the composite key: `report_id` + `page_name` + `visual_name`
- Visual metadata is in `lineage_report_visuals` with fields like `visual_name`, `type`, `title`

### Semantic Model Object Naming  
- The `list_semantic_model_objects()` function returns **object names** (not lineage tags)
- We join reports to semantic models by **name** first
- Then we **enrich** with lineage tags from the semantic model metadata tables
- This approach provides the foundation for downstream lineage calculations

### Design Philosophy
This notebook creates foundational views that link reports to their semantic model objects at the REPORT level. Visual-level lineage may require additional parsing of the `report_source` fields or joining with `lineage_report_visual_objects` (which tracks objects per visual). Complete lineage traversal (showing dependencies between measures, columns, and calculated fields) is performed in downstream processing to keep this notebook focused and maintainable.